### Project -

##### Setup

In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")

# import os
# os.getcwd()

# os.chdir('/content/drive/MyDrive/projects/AIFFEL_quest_eng/NLP/NLP05')


In [5]:
# %reload_ext autoreload
# %autoreload 2

import os
import json
import logging
import copy
from copy import deepcopy
import random
import functools
from typing import Optional, Dict, Sequence, List
from dataclasses import dataclass

import torch
import torch.nn as nn
from torch.utils.data import Dataset
import pandas as pd
import numpy as np

import transformers
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    PreTrainedTokenizerFast,
    GPT2Config,
    GPT2Model,
    pipeline
)

from chatgpt.dataset import RewardDataset
from chatgpt.models.base import RewardModel
from chatgpt.trainer.strategies import NaiveStrategy
from chatgpt.trainer.rm import RewardModelTrainer
from chatgpt.models.gpt import GPTActor, GPTCritic
from chatgpt.trainer import PPOTrainer

import guide

#### BaseLine Train

In [ ]:
try:
    root_path = os.path.dirname(os.path.abspath(__file__))
except:
    root_path = os.getcwd()

cfg = {
    "model_name": "skt/kogpt2-base-v2",
    "device": torch.device("xpu" if torch.xpu.is_available() else "cuda" if torch.cuda.is_available() else "cpu"),
    "root_path": root_path,
    "sft_output_dir": root_path + "/test",
    "sft_saved_dir": root_path + "/models/output_1_SFT",
    "sft_num_train_epochs": 1,
    "sft_per_device_train_batch_size": 4,
    "sft_per_device_eval_batch_size": 4,
    "sft_warmup_steps": 5,
    "sft_prediction_loss_only": True,
    "sft_fp16": False 
}

# tokenizer, model 준비
model = AutoModelForCausalLM.from_pretrained(cfg["model_name"]).to(cfg["device"])
tokenizer = PreTrainedTokenizerFast.from_pretrained(
    cfg["model_name"],
    bos_token='</s>', eos_token='</s>', unk_token='<unk>',
    pad_token='<pad>', mask_token='<mask>',
    padding_side="right",
    model_max_length=512,
)

# guide.show_base_model_and_dataset(cfg, model, tokenizer)
# guide.show_sft_and_rm_dataset(cfg)
guide.run_sft(cfg, model, tokenizer)
guide.run_reward_model(cfg)
guide.run_ppo(cfg, model, tokenizer)

#### Base Line BM Result
* 정량평가 
    * lm-eval --model hf \
        --model_args pretrained=models/output_1_SFT,tokenizer=skt/kogpt2-base-v2,dtype="float16" \
        --tasks kobest_copa,kobest_hellaswag,kobest_boolq \
        --batch_size auto \
        --limit 500


    * output_1_SFT

|     Tasks      |Version|Filter|n-shot| Metric |   |Value |   |Stderr|
|----------------|------:|------|-----:|--------|---|-----:|---|------|
|kobest_boolq    |      1|none  |     0|acc     |↑  |0.5340|±  |0.0223|
|                |       |none  |     0|f1      |↑  |0.3481|±  |   N/A|
|kobest_copa     |      1|none  |     0|acc     |↑  |0.5080|±  |0.0224|
|                |       |none  |     0|f1      |↑  |0.5077|±  |   N/A|
|kobest_hellaswag|      1|none  |     0|acc     |↑  |0.2440|±  |0.0192|
|                |       |none  |     0|acc_norm|↑  |0.2740|±  |0.0200|
|                |       |none  |     0|f1      |↑  |0.2427|±  |   N/A|

    * output_3_PPO

|     Tasks      |Version|Filter|n-shot| Metric |   |Value |   |Stderr|
|----------------|------:|------|-----:|--------|---|-----:|---|------|
|kobest_boolq    |      1|none  |     0|acc     |↑  |0.5340|±  |0.0223|
|                |       |none  |     0|f1      |↑  |0.3481|±  |   N/A|
|kobest_copa     |      1|none  |     0|acc     |↑  |0.5140|±  |0.0224|
|                |       |none  |     0|f1      |↑  |0.5134|±  |   N/A|
|kobest_hellaswag|      1|none  |     0|acc     |↑  |0.2380|±  |0.0191|
|                |       |none  |     0|acc_norm|↑  |0.2820|±  |0.0201|
|                |       |none  |     0|f1      |↑  |0.2371|±  |   N/A|

In [10]:
!lm-eval --model hf \
    --model_args pretrained=models/output_1_SFT,tokenizer=skt/kogpt2-base-v2,dtype="float16" \
    --tasks kobest_copa,kobest_hellaswag,kobest_boolq \
    --batch_size auto \
    --limit 500

2026-03-14:11:19:11 WARNING  [config.evaluate_config:281] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-03-14:11:19:19 INFO     [_cli.run:376] Selected Tasks: ['kobest_copa', 'kobest_hellaswag', 'kobest_boolq']
2026-03-14:11:19:21 INFO     [evaluator:211] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-03-14:11:19:21 INFO     [evaluator:236] Initializing hf model, with arguments: {'pretrained': 'models/output_1_SFT', 'tokenizer': 'skt/kogpt2-base-v2', 'dtype': 'float16'}
2026-03-14:11:19:26 INFO     [models.huggingface:161] Using device 'cuda:0'
2026-03-14:11:19:29 INFO     [models.huggingface:423] Model parallel was set to False, max memory was not set, and device map was set to {'': 'cuda:0'}
Loading weights: 100% 149/149 [00:00<00:00, 153.08it/s, Materializing param=transformer.wte.weight]
The tied weights mapping and config for this model specifies t

In [11]:
!lm-eval --model hf \
    --model_args pretrained=models/output_3_PPO,tokenizer=skt/kogpt2-base-v2,dtype="float16" \
    --tasks kobest_copa,kobest_hellaswag,kobest_boolq \
    --batch_size auto \
    --limit 500

2026-03-14:11:20:04 WARNING  [config.evaluate_config:281] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-03-14:11:20:30 INFO     [_cli.run:376] Selected Tasks: ['kobest_copa', 'kobest_hellaswag', 'kobest_boolq']
2026-03-14:11:20:35 INFO     [evaluator:211] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-03-14:11:20:35 INFO     [evaluator:236] Initializing hf model, with arguments: {'pretrained': 'models/output_3_PPO', 'tokenizer': 'skt/kogpt2-base-v2', 'dtype': 'float16'}
2026-03-14:11:20:52 INFO     [models.huggingface:161] Using device 'cuda:0'
2026-03-14:11:20:56 INFO     [models.huggingface:423] Model parallel was set to False, max memory was not set, and device map was set to {'': 'cuda:0'}
Loading weights: 100% 149/149 [00:03<00:00, 39.28it/s, Materializing param=transformer.wte.weight]
The tied weights mapping and config for this model specifies to